# 3. Evaluation, tables and resources

Evaluates trained runs (and the authors' BEVPlace++ checkpoint) without retraining. One protocol for every method
(the BEVPlace++ evaluation code): in each sequence the map is the frames before the split point, the queries start
200 frames after it, a match is correct within 5 m; Recall@1 and Recall@1%.

Evaluation types (`EVAL_TYPES` in `src/lpr/evals.py`, run by `scripts/evaluate.py`):

| type | what | methods |
|---|---|---|
| `seqs_heading` | recall on 00, 02, 05, 06, 08, and per heading group of the queries (same / crossing / opposite) | all |
| `tnet` | scans rotated about z: is the T-Net following the rotation, is the descriptor still the same scan | `ndt` |
| `compass` | every scan rotated to one heading by its pose; queries with a compass error N(0, sigma^2) | all |
| `rotation` | map as recorded, every query rotated by its own angle from U[-alpha, alpha] (rotation curve) | all |
| `sparsity` | 32 / 16 beams: `both` (map and queries sparse) and `query` (64-beam map, sparse queries) | all |

Each result is kept in `<run>/eval/<type>/<checkpoint>/` and is not computed again once complete, so Run all again
after a disconnect. `scripts/summarize_runs.py` collects everything into `MyDrive/lpr/runs_summary.csv` and
`MyDrive/lpr/evals_summary.csv`, the source of the report's tables.

Optional: the density check (section 6), the 08 split of the BEVPlace++ paper (section 7), and the resource
benchmark with the method figures (section 8). Runtime: GPU (T4 is enough). Set cell 1, then **Runtime -> Run all**.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/dov85/lidar-place-recognition"
CODE_ON_DRIVE = ""                      # or the path of this code folder copied to Drive (then REPO_URL is not used)
CODE = "/content/lpr"                   # the code on this machine
DRIVE = "/content/drive/MyDrive/lpr"    # everything that is kept: variants, runs, evaluations, tables
KITTI = "/content/kitti"                # poses (and raw scans when needed)

METHODS = ["ndt", "myndt", "bev"]       # which kinds of runs
STATUS = ["finished"]                   # runs with this status in their config.json
ONLY_RUNS = []                          # if not empty: only these run folders (any status), e.g. ["2026-10-20_3fe7a5"]
EXCLUDE_RUNS = []
EVAL_CKPT = "last"                      # "last" or a checkpoint file name, the same for every run
OFFICIAL_BEV = True                     # also the authors' BEVPlace++ checkpoint, on the default BEV variant
EVAL_TYPES = ["seqs_heading", "compass", "rotation", "sparsity", "tnet"]
PARAMS = {}                             # overrides per type, e.g. {"sparsity": {"beams": [16]}}

DENSITY_CHECK = False                   # section 6
CHECK_RUNS = []                         # runs of the density check ([] = all the runs above)
SPLIT08_CHECK = False                   # section 7
BENCHMARK = False                       # section 8
BENCH_SEQ, N_SCANS, FIG_IDX = "08", 50, 30

## 2. Drive, code, poses

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
import os, sys, json, glob, subprocess, importlib
os.makedirs(DRIVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU"
if not os.path.exists(CODE):
    if CODE_ON_DRIVE:
        subprocess.run(["cp", "-r", CODE_ON_DRIVE, CODE], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, CODE], check=True)
elif os.path.isdir(f"{CODE}/.git"):
    subprocess.run(["git", "-C", CODE, "pull", "-q"])
%cd {CODE}
!bash scripts/setup_third_party.sh
!pip install -q -r requirements.txt
sys.path.insert(0, f"{CODE}/src")
import lpr.experiments as ex, lpr.evals as ev, lpr.colab as cl
for m in (ex, ev, cl):
    importlib.reload(m)
try:                                    # container SAS URL (read + list) in Colab Secrets; never in a notebook
    SAS = userdata.get("AZURE_SAS")
except Exception:
    SAS = None
    print("no AZURE_SAS in Colab Secrets: only data that is already on Drive is used")
missing_poses = [s for s in ev.ALL_SEQS if not os.path.exists(f"{KITTI}/poses/{s}.txt")]
if missing_poses:
    !python scripts/download_kitti.py --out {KITTI} --seqs {" ".join(missing_poses)} --poses_only

## 3. The runs

In [ ]:
import pandas as pd
OFFICIAL_CKPT = f"{CODE}/third_party/BEVPlace2/runs/Aug08_10-17-29/model_best.pth.tar"
DATA = {"ndt": "ndt", "myndt": "ndt", "bev": "bev"}     # whose variants a method uses
if OFFICIAL_BEV and "bev" in METHODS:                    # the authors' checkpoint as a run of its own
    bev_base = ex.resolve_variant(DRIVE, "bev", {})[0]
    off = f"{DRIVE}/runs/bev/official_{bev_base}"
    if not os.path.exists(f"{off}/config.json"):
        os.makedirs(off, exist_ok=True)
        ex.write_json(f"{off}/config.json", {"run": os.path.basename(off), "method": "bev", "bev_variant": bev_base,
                      "status": "finished", "params": {"source": "official checkpoint Aug08_10-17-29/model_best.pth.tar"}})
runs = []
for m in METHODS:
    for cfg_path in sorted(glob.glob(f"{DRIVE}/runs/{m}/*/config.json")):
        run_dir = os.path.dirname(cfg_path)
        name = os.path.basename(run_dir)
        cfg = json.load(open(cfg_path))
        official = m == "bev" and name.startswith("official_")
        if name in EXCLUDE_RUNS or (official and not OFFICIAL_BEV) or (ONLY_RUNS and name not in ONLY_RUNS):
            continue
        if not ONLY_RUNS and cfg.get("status") not in STATUS and not official:
            continue
        if not official and not glob.glob(f"{run_dir}/epoch_*.pth"):
            continue
        runs.append(dict(method=m, run=name, run_dir=run_dir, ckpt=OFFICIAL_CKPT if official else EVAL_CKPT,
                         variant=cfg.get(f"{DATA[m]}_variant"), data=DATA[m]))
RUNS = pd.DataFrame(runs)
display(RUNS[["method", "run", "variant"]])

## 4. Data and evaluations
Every needed sequence of every variant on this machine (from Drive, or the blob container), with the 32 / 16-beam
variants for `sparsity` and the density check; then each type on each run.

In [ ]:
import time
SP = {e: ev.full_eval_params(e, PARAMS.get(e, {})) for e in EVAL_TYPES}
SEQS = sorted({s for p in SP.values() for s in p.get("seqs", [p.get("seq")]) if s})
BEAMS = SP["sparsity"]["beams"] if "sparsity" in SP else ([16] if DENSITY_CHECK else [])
for (dm, variant), _ in RUNS.groupby(["data", "variant"]):
    cl.to_machine(DRIVE, dm, variant, SEQS, SAS)
    cl.sparse_to_machine(DRIVE, dm, variant, BEAMS, SEQS, SAS)
for e in EVAL_TYPES:
    for r in RUNS.itertuples():
        if r.method not in ev.EVAL_TYPES[e]["methods"]:
            continue
        t0 = time.time()
        print(f"===== {e}: {r.method} {r.run}")
        cl.run(["scripts/evaluate.py", "--method", r.method, "--run_dir", r.run_dir, "--ckpt", r.ckpt, "--eval_type", e,
                "--data_dir", f"/content/{r.data}/{r.variant}", "--kitti_root", KITTI, "--params", json.dumps(PARAMS.get(e, {}))],
               tail=8)
        print(f"({(time.time() - t0) / 60:.1f} min)")

## 5. Tables (Recall@1, %)

In [ ]:
!python scripts/summarize_runs.py --drive {DRIVE} > /dev/null
E = pd.read_csv(f"{DRIVE}/evals_summary.csv", dtype={"seq": str, "run": str, "checkpoint": str})
E = E[E.run.isin(RUNS.run)]
OTHER = ["00", "02", "05", "06"]
for e in EVAL_TYPES:
    d = E[E.eval_type == e].copy()
    if d.empty:
        continue
    print(f"\n#################### {e}")
    if e == "seqs_heading":
        t = d.pivot_table(index=["method", "run"], columns="seq", values="R@1")
        t["mean"] = t.mean(axis=1)
        display(t.round(1))
        print("08: Recall@1 per heading group (share of the queries in the % columns)")
        g = d[d.seq == "08"].set_index(["method", "run"])
        display(g[[c for c in g.columns if c.startswith("% ") or c.startswith("R@1 ")]].round(1))
    elif e in ("compass", "rotation"):
        key = {"compass": "sigma_deg", "rotation": "alpha_deg"}[e]
        print("08"); display(d[d.seq == "08"].pivot_table(index=["method", "run"], columns=key, values="R@1").round(1))
        print("mean of 00, 02, 05, 06"); display(d[d.seq.isin(OTHER)].pivot_table(index=["method", "run"], columns=key, values="R@1").round(1))
    elif e == "sparsity":
        d["k"] = d["mode"] + " " + d["beams"].astype(int).astype(str)
        print("08"); display(d[d.seq == "08"].pivot_table(index=["method", "run"], columns="k", values="R@1").round(1))
        print("mean of 00, 02, 05, 06"); display(d[d.seq.isin(OTHER)].pivot_table(index=["method", "run"], columns="k", values="R@1").round(1))
    elif e == "tnet":
        display(d.dropna(axis=1, how="all").set_index(["run", "angle"])
                 .drop(columns=["method", "eval_type", "complete", "seq", "checkpoint"], errors="ignore").round(3))

## 6. Optional (`DENSITY_CHECK`): 16-beam against 64-beam descriptors of the same scans
`scripts/density_check.py`, on 400 scans of 08 and of 00: `self_top1` (% of scans whose 16-beam descriptor is
closest to the 64-beam descriptor of the same scan) and `spread` (median distance between 16-beam descriptors of
different scans / the same for 64 beams; well below 1 = they collapse together). Kept in `<run>/eval/density_check_<seq>.json`.

In [ ]:
if not DENSITY_CHECK:
    print("skipped (DENSITY_CHECK = False)")
else:
    rows = []
    for r in RUNS[RUNS.run.isin(CHECK_RUNS) if CHECK_RUNS else slice(None)].itertuples():
        for s in ["08", "00"]:
            out = f"{r.run_dir}/eval/density_check_{s}.json"
            if not os.path.exists(out):
                os.makedirs(os.path.dirname(out), exist_ok=True)
                cl.run(["scripts/density_check.py", "--method", r.method, "--run_dir", r.run_dir, "--ckpt", r.ckpt,
                        "--data_dir", f"/content/{r.data}/{r.variant}", "--kitti_root", KITTI, "--seq", s, "--out", out], tail=3)
            if os.path.exists(out):
                rows += json.load(open(out))
    DC = pd.DataFrame(rows)
    DC.to_csv(f"{DRIVE}/density_check.csv", index=False)
    display(DC[DC.beams == 16].pivot_table(index=["method", "run"], columns="seq", values=["self_top1", "spread"]).round(2))

## 7. Optional (`SPLIT08_CHECK`): the authors' BEVPlace++ checkpoint with the 08 split of the paper
The official code splits 08 at frame 1000 (map 0-999), Table II of the paper at 3000 (map 0-2999).

In [ ]:
if not SPLIT08_CHECK:
    print("skipped (SPLIT08_CHECK = False)")
else:
    off = RUNS[RUNS.run.str.startswith("official_")].iloc[0]
    rows = []
    for split in [1000, 3000]:
        out = f"{off.run_dir}/eval/split08/split_{split}.json"
        if not os.path.exists(out):
            os.makedirs(os.path.dirname(out), exist_ok=True)
            cl.run(["scripts/eval_bev_kitti.py", "--ckpt", off.ckpt, "--bev_dir", f"/content/bev/{off.variant}",
                    "--kitti_root", KITTI, "--seq", "08", "--split", split, "--out", out], tail=3)
        r = json.load(open(out))
        rows.append({"split": split, **{k: r[k] for k in r if k.startswith("R@") or k.startswith("n_queries")}})
    display(pd.DataFrame(rows))

## 8. Optional (`BENCHMARK`): resources and the method figures
`scripts/benchmark.py` on the first `N_SCANS` scans of `BENCH_SEQ`: time per scan of every preprocessing step on the
CPU (crop + ground removal, G with and without the ground, BEV image), stored size per scan, and for every network
(random weights) the time per descriptor, the peak GPU memory of inference and of one training step, and the number
of parameters (`comparison.csv`). `scripts/figure_method.py`: one scan as points, NDT cells and BEV image;
`scripts/figure_networks.py`: the three networks with the output shape of every stage. Results: `MyDrive/lpr/benchmark/`.

In [ ]:
from IPython.display import Image, display
if not BENCHMARK:
    print("skipped (BENCHMARK = False)")
else:
    OUT = f"{DRIVE}/benchmark"
    os.makedirs(OUT, exist_ok=True)
    vel = f"{KITTI}/sequences/{BENCH_SEQ}/velodyne"
    if len(os.listdir(vel) if os.path.isdir(vel) else []) < N_SCANS:
        !python scripts/download_kitti.py --out {KITTI} --seqs {BENCH_SEQ} --first {N_SCANS} --threads 16
    !python scripts/benchmark.py --kitti_root {KITTI} --seq {BENCH_SEQ} --n {N_SCANS} --out_dir {OUT}
    display(pd.read_csv(f"{OUT}/comparison.csv").set_index("network").T)
    !python scripts/figure_method.py --kitti_root {KITTI} --seq {BENCH_SEQ} --idx {FIG_IDX} --out {OUT}/fig_method.png
    !python scripts/figure_networks.py --out {OUT}/fig_networks.png
    display(Image(f"{OUT}/fig_method.png", width=1300)); display(Image(f"{OUT}/fig_networks.png", width=1300))